# I22 chunk processing with server-side `HDFSource` slicing

This example keeps the scheduling loop in the notebook while the MoDaCor runtime reads each requested detector and normalization slice directly from HDF5. A small buffer-backed pilot establishes the reduced output schema; the actual chunk workload uses typed `HDFSource` bindings.


## Configuration

The defaults reproduce the full four-measurement, ten-chunk validation. For a quick smoke run, use `MEASUREMENT_LIMIT = 1` and `FRAME_COUNT = 20`.


In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "example_utils.py").is_file():
        EXAMPLES_ROOT = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter from MoDaCor_examples or one of its subdirectories.")

sys.path.insert(0, str(EXAMPLES_ROOT))
from example_utils import locate_example_dir

PROJECT_DIR = locate_example_dir("DLS/I22")
sys.path.insert(0, str(PROJECT_DIR))

import atexit
import sys

from IPython.display import JSON, display

import hdf5plugin
import modacor
from modacor.client import LocalRuntimeServer

from i22_helpers import (
    build_complete_plan,
    chunk_spec,
    chunk_work_items,
    compare_run_groups,
    direct_sample_registration,
    prepare_inputs,
    sample_aligned_paths,
    source_registrations,
    trace_source_slices,
    upload_sample_chunk,
    validate_chunk_sources,
    validation_pipeline_yaml,
)


In [ ]:
DETECTORS = ("SAXS", "WAXS")
MEASUREMENT_LIMIT = 4
FRAME_COUNT = 100
CHUNK_SIZE = 10
COMPRESSION = "gzip"
COMPRESSION_LEVEL = 1
# Provisional scalar retained from the DAWN processing record for this example.
ABSOLUTE_INTENSITY_FACTOR = 3.8e-15
BEAMLINE_CONFIGURATION = "usaxs_saxs_waxs"  # special nosecone; no aluminium attenuator
# Unobstructed-beam measurement used to calibrate bsdiodes against I0.
TRANSMISSION_REFERENCE_FILE = PROJECT_DIR / "data" / "i22-977723.nxs"
OVERWRITE_PREPROCESSED = False

SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8901
TRACE = {"enabled": True, "watch": {"sample": ["signal"], "background": ["signal"]}}
WORK_DIR = PROJECT_DIR / "work" / "chunk_server"
OUTPUT_DIR = WORK_DIR


## Prepare the shared inputs and chunk grid

Preprocessing is lean and cached. The explicit plan is derived later from one real pilot result because this pipeline reduces both input batch dimensions.


In [ ]:
inputs = prepare_inputs(
    PROJECT_DIR,
    beamline_configuration=BEAMLINE_CONFIGURATION,
    transmission_reference_file=TRANSMISSION_REFERENCE_FILE,
    absolute_intensity_factor=ABSOLUTE_INTENSITY_FACTOR,
    overwrite=OVERWRITE_PREPROCESSED,
)

print(f"Python: {sys.executable}")
print(f"MoDaCor: {modacor.__version__}")
print(f"Measurements: {len(inputs.measurements)}")
print(f"Preprocessed data: {inputs.work_dir / 'preprocessed'}")

measurements = inputs.measurements[:MEASUREMENT_LIMIT]
work_items = chunk_work_items(measurements, frame_count=FRAME_COUNT, chunk_size=CHUNK_SIZE)
source_shapes = {
    detector: validate_chunk_sources(measurements, detector, frame_count=FRAME_COUNT)
    for detector in DETECTORS
}
WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"Workload: {len(measurements)} measurements × {len(work_items) // len(measurements)} chunks × {len(DETECTORS)} detectors")


## Start or reuse the MoDaCor runtime


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
server = LocalRuntimeServer(
    host=SERVER_HOST,
    port=SERVER_PORT,
    log_path=OUTPUT_DIR / "modacor_server.log",
    environment={"HDF5_PLUGIN_PATH": hdf5plugin.PLUGINS_PATH},
)
client = server.start()
atexit.register(server.stop)
print(f"{'Started' if server.launched else 'Reusing'} runtime at {client.base_url}")


## Run the HDFSource workload


In [ ]:
output_path = WORK_DIR / f"i22_{len(measurements)}x{FRAME_COUNT}_hdf_chunks.h5"
summaries = []

for detector in DETECTORS:
    session = client.replace_session(
        f"i22-{detector.lower()}-hdf-chunks",
        name=f"I22 {detector} HDFSource chunks",
        pipeline_yaml=validation_pipeline_yaml(inputs.pipeline_paths[detector]),
        trace=TRACE,
    )
    # A buffer-backed pilot establishes the schema without loading a complete detector stack.
    session.register_sources(
        *source_registrations(inputs),
        {"ref": "sample", "type": "buffer", "location": "buffer://session"},
    )
    first = work_items[0]
    upload_sample_chunk(session.source_buffer("sample"), detector, first.source_path, first.start, first.stop)
    pilot_name = f"{first.master_path.stem}_{detector.lower()}_pilot"
    pilot_path = WORK_DIR / f"{pilot_name}.h5"
    pilot_path.unlink(missing_ok=True)
    session.process(
        mode="full",
        run_name=pilot_name,
        rollback_snapshot=False,
        write_hdf={"path": str(pilot_path), "data_paths": ["/sample/signal"]},
    )

    plan = build_complete_plan(
        pilot_path, pilot_name, detector, measurements, source_shapes[detector],
        frame_count=FRAME_COUNT, chunk_size=CHUNK_SIZE, source_mode="hdf",
    )
    output = client.chunked_outputs.create(
        sink={
            "ref": f"i22_{detector.lower()}_hdf_result",
            "type": "hdf_chunked",
            "location": str(output_path),
            "kwargs": {"compression": COMPRESSION, "compression_opts": COMPRESSION_LEVEL},
        },
        subpath=f"processed_{detector.lower()}_hdf_chunks",
        plan=plan,
        collision="replace",
    )

    current_source = None
    for completed, item in enumerate(work_items, start=1):
        if item.source_path != current_source:
            session.register_source(direct_sample_registration("hdf", item.source_path))
            current_source = item.source_path
        spec = chunk_spec(plan, item)
        session.process(
            mode="partial",
            run_name=f"{item.master_path.stem}_{detector.lower()}_hdf_{item.start:03d}_{item.stop:03d}",
            rollback_snapshot=False,
            chunk_output=output.chunk(spec),
        )
        if completed == 1 or completed % 5 == 0 or completed == len(work_items):
            print(f"{detector} {completed:02d}/{len(work_items)}: {spec.chunk_id}")

    final = output.finalize()
    bound_paths = trace_source_slices(output_path, plan)
    if bound_paths != set(sample_aligned_paths(detector)):
        raise AssertionError(f"Unexpected source bindings: {sorted(bound_paths)}")
    summaries.append({"detector": detector, "status": final["status"], "chunks": final["completed_chunks"]})
    session.delete()

display(JSON({"output": str(output_path), "runs": summaries}))


## Optional cross-transport comparison


In [ ]:
buffer_path = WORK_DIR / f"i22_{len(measurements)}x{FRAME_COUNT}_buffer_chunks.h5"
if buffer_path.is_file():
    comparisons = {
        detector: compare_run_groups(
            buffer_path, f"processed_{detector.lower()}_buffer_chunks",
            output_path, f"processed_{detector.lower()}_hdf_chunks",
            measurement_count=len(measurements),
            chunks_per_measurement=len(work_items) // len(measurements),
        )
        for detector in DETECTORS
    }
    display(JSON(comparisons))
else:
    print("Run I22_solids_chunked_buffer.ipynb for a cross-transport comparison.")


## Cleanup


In [ ]:
server.stop()
print("Stopped the notebook-owned runtime." if not client.is_ready() else "Left the external runtime running.")
